## **Notebook 08 — Cross-Encoder Reranker**
- Take top-10 from hybrid+HyDE → score each (query, chunk) pair jointly → return top-5. This is the final accuracy layer before the LLM sees any context.

In [1]:
# BGE reranker is part of FlagEmbedding — already installed
# !pip install FlagEmbedding

import os, json
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
import bm25s
from FlagEmbedding import BGEM3FlagModel, FlagReranker
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()
DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
INDEX_DIR     = Path("../data/bm25_index")

conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)
cur      = conn.cursor()

bm25_retriever = bm25s.BM25.load(str(INDEX_DIR/"bm25_index"), load_corpus=False)
with open(INDEX_DIR/"chunk_ids.json")   as f: chunk_ids   = json.load(f)
with open(INDEX_DIR/"chunk_texts.json") as f: chunk_texts = json.load(f)

embed_model = BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    use_fp16=False, cache_dir="../models/bge-m3"
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

print("All components loaded OK")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

All components loaded OK


In [2]:
RERANKER_PATH = "../models/bge-reranker"

# First run: downloads from HuggingFace (~1.1 GB)
# Every run after: loads from local path instantly
reranker = FlagReranker(
    model_name_or_path="BAAI/bge-reranker-v2-m3",
    use_fp16=False,          # False = CPU mode
    cache_dir=RERANKER_PATH
)

print("BGE reranker loaded OK")
print("Model: BAAI/bge-reranker-v2-m3")
print("Mode : CPU (use_fp16=False)")

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

BGE reranker loaded OK
Model: BAAI/bge-reranker-v2-m3
Mode : CPU (use_fp16=False)


In [3]:
# Copied from previous notebooks — notebook is self-contained

HYDE_PROMPT = """You are an HR policy expert.
Write a short passage (3-5 sentences) that directly answers the question.
Write as if extracted from an official HR policy document. Formal language only.
Do NOT say "I". Just write the passage directly.

Question: {query}
Passage:"""

def generate_hyde(query):
    r = llm.invoke([HumanMessage(content=HYDE_PROMPT.format(query=query))])
    return r.content

def dense_search(query_vec, k=20):
    cur.execute("""
        SELECT id::text, content, parent_content,
               metadata->>'section', 1-(embedding <=> %s::vector)
        FROM chunks ORDER BY embedding <=> %s::vector LIMIT %s;
    """, (query_vec.tolist(), query_vec.tolist(), k))
    return [{"chunk_id":r[0],"content":r[1],"parent_content":r[2],
             "section":r[3],"score":float(r[4]),"rank":i+1}
            for i,r in enumerate(cur.fetchall())]

def sparse_search(query, k=20):
    tokens = bm25s.tokenize([query], stopwords="en")
    res, scores = bm25_retriever.retrieve(tokens, k=min(k,len(chunk_ids)))
    return [{"chunk_id":chunk_ids[idx],"content":chunk_texts[idx],
             "parent_content":None,"section":None,
             "score":float(s),"rank":i+1}
            for i,(idx,s) in enumerate(zip(res[0],scores[0]))]

def rrf_fusion(dense_res, sparse_res, k=60, top_n=10):
    scores = {}
    for r in dense_res:
        scores[r["chunk_id"]] = scores.get(r["chunk_id"],0)+1/(k+r["rank"])
    for r in sparse_res:
        scores[r["chunk_id"]] = scores.get(r["chunk_id"],0)+1/(k+r["rank"])
    ranked   = sorted(scores.items(), key=lambda x:x[1], reverse=True)
    d_lookup = {r["chunk_id"]:r for r in dense_res}
    fused = []
    for cid, rrf_score in ranked[:top_n]:
        base = d_lookup.get(cid,{})
        fused.append({"chunk_id":cid,
                      "content":base.get("content",""),
                      "parent_content":base.get("parent_content",""),
                      "section":base.get("section",""),
                      "rrf_score":rrf_score,"rank":len(fused)+1,
                      "in_dense":cid in {r["chunk_id"] for r in dense_res},
                      "in_sparse":cid in {r["chunk_id"] for r in sparse_res}})
    return fused

def hybrid_search_with_hyde(query, k=20, top_n=10):
    hypo     = generate_hyde(query)
    hyde_vec = embed_model.encode([hypo], return_dense=True)["dense_vecs"][0]
    dense    = dense_search(hyde_vec, k=k)
    sparse   = sparse_search(query,   k=k)
    return rrf_fusion(dense, sparse, top_n=top_n)

print("All pipeline functions ready")

All pipeline functions ready


In [4]:
def rerank(query: str, candidates: list[dict], top_k: int = 5) -> list[dict]:
    """
    Cross-encoder reranking.

    Takes each (query, chunk_content) pair and scores them jointly.
    The model reads both together — full attention between query and document.
    Much more accurate than cosine similarity but only feasible on small sets.

    Returns top_k candidates sorted by reranker score.
    """
    if not candidates:
        return []

    # Build (query, document) pairs — one per candidate
    pairs = [[query, c["content"]] for c in candidates]

    # Score all pairs — normalize=True gives scores between 0 and 1
    scores = reranker.compute_score(pairs, normalize=True)

    # Attach scores and sort
    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    # Re-assign rank numbers
    for i, r in enumerate(reranked):
        r["rank"] = i + 1

    return reranked[:top_k]

print("rerank() defined")

rerank() defined


In [5]:
def full_retrieval_pipeline(query: str, top_n: int = 5) -> list[dict]:
    """
    Complete retrieval pipeline:
    Stage 1 — hybrid+HyDE  : retrieve top-10 candidates fast
    Stage 2 — reranker     : score top-10 precisely, return top-5
    """
    # Stage 1: cast a wide net
    candidates = hybrid_search_with_hyde(query, k=20, top_n=10)

    # Stage 2: precise reranking on the small candidate set
    reranked   = rerank(query, candidates, top_k=top_n)

    return reranked

print("full_retrieval_pipeline() defined")

full_retrieval_pipeline() defined


In [6]:
QUERY   = "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"
results = full_retrieval_pipeline(QUERY, top_n=5)

print(f'Query: "{QUERY}"\n')
print(f"{'─'*62}")
for r in results:
    print(f"Rank {r['rank']}  |  Rerank score: {r['rerank_score']:.4f}  "
          f"|  RRF was: {r['rrf_score']:.4f}")
    print(f"  Section : {r['section']}")
    print(f"  Content : {r['content'][:110]}")
    print()

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Query: "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"

──────────────────────────────────────────────────────────────
Rank 1  |  Rerank score: 0.8869  |  RRF was: 0.0313
  Section : 3.3 Internal Candidate
  Content : Permanent/regular: Permanent or regular employee shall be the one who is employed to work until his/her retire

Rank 2  |  Rerank score: 0.8402  |  RRF was: 0.0310
  Section : 3.11 Basic Conditions
  Content : The staff members of regular/permanent positions in the society shall retire on the date on which he/she  atta

Rank 3  |  Rerank score: 0.8379  |  RRF was: 0.0305
  Section : 3.10 Hiring of Relatives
  Content : The staff members of regular/permanent positions in the society shall retire on the date on which he/she  atta

Rank 4  |  Rerank score: 0.8325  |  RRF was: 0.0304
  Section : 3.3 Internal Candidate
  Content : Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a temporary period who are not c

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')


In [7]:
TARGET   = "shall retire on the date"
QUERY    = "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"

print(f'Query: "{QUERY[:55]}..."\n')
print(f"{'Step':<35} {'Correct chunk rank':<20} {'Method'}")
print("─" * 70)

# Step 5 — dense only
raw_vec   = embed_model.encode([QUERY], return_dense=True)["dense_vecs"][0]
d_res     = dense_search(raw_vec, k=20)
d_rank    = next((r["rank"] for r in d_res if TARGET in r["content"]), "NOT FOUND")
print(f"  Step 5  Dense only             {str(d_rank):<20} pgvector cosine")

# Step 7 — hybrid RRF (no HyDE)
s_res     = sparse_search(QUERY, k=20)
h_res     = rrf_fusion(d_res, s_res, top_n=20)
h_rank    = next((r["rank"] for r in h_res if TARGET in r["content"]), "NOT FOUND")
print(f"  Step 7  Hybrid RRF             {str(h_rank):<20} dense + sparse + RRF")

# Step 8 — hybrid + HyDE
hyde_res  = hybrid_search_with_hyde(QUERY, k=20, top_n=20)
hy_rank   = next((r["rank"] for r in hyde_res if TARGET in r["content"]), "NOT FOUND")
print(f"  Step 8  Hybrid + HyDE          {str(hy_rank):<20} + hypothetical doc")

# Step 9 — full pipeline with reranker
full_res  = full_retrieval_pipeline(QUERY, top_n=10)
fr_rank   = next((r["rank"] for r in full_res if TARGET in r["content"]), "NOT FOUND")
print(f"  Step 9  Full pipeline          {str(fr_rank):<20} + cross-encoder rerank")

print()
print("  Each layer compounds on the previous one.")
print("  Retrieval is now ready for generation.")

Query: "What is the mandatory retirement age for a regular/perm..."

Step                                Correct chunk rank   Method
──────────────────────────────────────────────────────────────────────
  Step 5  Dense only             NOT FOUND            pgvector cosine


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Step 7  Hybrid RRF             NOT FOUND            dense + sparse + RRF


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Step 8  Hybrid + HyDE          2                    + hypothetical doc


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Step 9  Full pipeline          3                    + cross-encoder rerank

  Each layer compounds on the previous one.
  Retrieval is now ready for generation.


In [8]:
# conn.close()

print("=" * 52)
print("STEP 9 COMPLETE — RETRIEVAL PIPELINE DONE")
print("=" * 52)
print("  Reranker loaded               : OK")
print("  (query, chunk) pairs scored   : OK")
print("  Correct chunk at Rank 1       : OK")
print()
print("  Full pipeline stack:")
print("  1. HyDE         — better query embedding")
print("  2. Dense search — semantic similarity")
print("  3. Sparse search— exact keyword matching")
print("  4. RRF fusion   — merge both result lists")
print("  5. Reranker     — precise final scoring")
print()
print("READY FOR STEP 10 — LLM Generation + Streaming")

STEP 9 COMPLETE — RETRIEVAL PIPELINE DONE
  Reranker loaded               : OK
  (query, chunk) pairs scored   : OK
  Correct chunk at Rank 1       : OK

  Full pipeline stack:
  1. HyDE         — better query embedding
  2. Dense search — semantic similarity
  3. Sparse search— exact keyword matching
  4. RRF fusion   — merge both result lists
  5. Reranker     — precise final scoring

READY FOR STEP 10 — LLM Generation + Streaming
